In [16]:
import sys
from pathlib import Path
current_dir = Path().resolve()
sys.path.append(current_dir.parent.parent.as_posix())

import os
from os import listdir
from os.path import isfile, join

# %% General setup
import pandas as pd
from pathlib import Path
from data_io import DataIO

import utils
import numpy as np
import plotly
import plotly.graph_objs as go
from utils import make_figure, save_fig
from scipy.stats import wilcoxon
from audrey.preprocessing.project_colors import ProjectColors

# Load data
session_id = '251015'
data_dir = Path(r'/media/aleong/Audrey-experiments/dataset')
figure_dir = data_dir / Path(session_id)

if not os.path.exists(figure_dir):
    os.makedirs(figure_dir)
    print(f"Created the figure folder : {figure_dir}")

data_io = DataIO(data_dir)
loadname = data_dir / f'{session_id}_cells.csv'
data_io.load_session(session_id, load_pickle=False, load_waveforms=False)
data_io.dump_as_pickle()

cells_df = pd.read_csv(loadname, header=[0, 1], index_col=0)
clrs = ProjectColors()

# INCLUDE_RANGE = 50  # include cells at max distance = 50 um

### Select the recordings to analyse

In [17]:
# Print available recording ids
print("Available recording ids:")
for rec_id in data_io.recording_ids:
    print(f"- {rec_id}")


# Manually select recordings to analyse
recording_nrs = [6, 7]

selected_rec_names = []
for r in recording_nrs:
    rec_name = None
    for rec_id in data_io.recording_ids:
        if f'_{r:03d}_' in rec_id:
            rec_name = rec_id
            selected_rec_names.append(rec_name)
            break

print(f"\nSelected recordings:")
for rec_name in selected_rec_names:
    print(f"- {rec_name}")

Available recording ids:
- 251015_A_005_noblocker_light_series0001
- 251015_A_006_noblocker_pa_prr_series
- 251015_A_007_noblocker_light_pa_series

Selected recordings:
- 251015_A_006_noblocker_pa_prr_series
- 251015_A_007_noblocker_light_pa_series


### Detect the electrode stimulation site with the most significant responses, per cell

In [18]:
# %% Detect electrode stim site with most significant responses, per cell
pref_ec_dict = {}

for cluster_id in data_io.cluster_df.index.values:
    pref_ec = None
    n_sig_pref_ec = None

    max_fr = None
    for ec in data_io.burst_df.electrode.unique():
        if pd.isna(ec):
            continue

        df = data_io.burst_df.query(f'electrode == {float(ec)}')
        tids = df.train_id.unique()
        n_sig = 0
        for tid in tids:
            if cells_df.loc[cluster_id, (tid, 'is_significant')] is True:
                n_sig += 1

        if n_sig > 1:
            if pref_ec is None or n_sig > n_sig_pref_ec:
                pref_ec = ec
                n_sig_pref_ec = n_sig

    pref_ec_dict[cluster_id] = pref_ec


print('Electrode stimulation site with the most significant responses per cell detected.\n\n------ End Of Cell ------')

Electrode stimulation site with the most significant responses per cell detected.

------ End Of Cell ------


### Plot the raster plots for each cell

In [ ]:
##%% Plot raster plots for each individual cell

cluster_ids = data_io.cluster_df.index.values

electrodes  = data_io.burst_df.electrode.unique()

print(f'saving data in: {figure_dir / "raster plots"}')

for cluster_id in cluster_ids:
    cluster_data = utils.load_obj(data_dir / 'bootstrapped' / f'bootstrap_{cluster_id}.pkl')

    n_electrodes = electrodes.size

    ec = pref_ec_dict[cluster_id]
    if ec is None:    # Skip clusters without preferred electrode
        print(f"None, cluster {cluster_id}")
        continue

    # Setup figure layout
    fig = utils.make_figure(
        width    =1,
        height   =1.5,
        x_domains={
            1: [[0.15, 0.95]],
        },
        y_domains={
            1: [[0.1, 0.9]]
        },
    )

    # Setup variables for plotting
    burst_offset   = 0
    x_plot, y_plot = [-500, 500, None], [0, 0, None]
    x_lines, y_lines = [], []
    yticks         = []
    ytext          = []
    pos            = dict(row=1, col=1)

    has_sig = False

    for rec_i, rec_name in enumerate(selected_rec_names):
        d_select = data_io.burst_df.query('electrode == @ec and '
                                              'recording_name == @rec_name').copy()
        d_select.sort_values('duty_cycle', inplace=True)

        # repetition_frequencies = d_select.repetition_frequency.unique()

        if '_light_pa_' in rec_name:
        #if 'light-prr' in rec_name:
            duty_cycles = d_select.laser_duty_cycle.unique()
        elif '_pa_' in rec_name:
        #elif 'prr' in rec_name:
            duty_cycles = d_select.duty_cycle.unique()


        for dc in duty_cycles:
            if '_light_pa_' in rec_name:
            #if 'light-prr' in rec_name:
                tid   = d_select.query('laser_duty_cycle == @dc').iloc[0].train_id
                frep  = data_io.burst_df.query('train_id == @tid').iloc[0].laser_duty_cycle
                bd    = data_io.burst_df.query('train_id == @tid').iloc[0].laser_burst_duration
                rname = 'PA+light'
            elif '_pa_' in rec_name:
            #elif 'prr' in rec_name:
                tid   = d_select.query('duty_cycle == @dc').iloc[0].train_id
                frep  = data_io.burst_df.query('train_id == @tid').iloc[0].duty_cycle
                bd    = data_io.burst_df.query('train_id == @tid').iloc[0].burst_duration
                rname = 'PA'

        
            spike_times = cluster_data[tid]['spike_times']
            bins = cluster_data[tid]['bins']

            ytext.append(f'dc: {frep:.0f} bd: {bd:.0f}, {rec_i}, {rname}')
            yticks.append(burst_offset + len(spike_times) / 2)

            for burst_i, sp in enumerate(spike_times):
                x_plot.append(np.vstack([sp, sp, np.full(sp.size, np.nan)]).T.flatten())
                y_plot.append(np.vstack([np.ones(sp.size) * burst_offset,
                                         np.ones(sp.size)* burst_offset +1, np.full(sp.size, np.nan)]).T.flatten())
                burst_offset += 1

        x_lines.extend([-500, 500, None])
        y_lines.extend([burst_offset, burst_offset, None])

    x_plot = np.hstack(x_plot)
    y_plot = np.hstack(y_plot)

    fig.add_scatter(
        x=x_lines, y=y_lines,
        mode='lines', line=dict(color='blue', dash='3px, 3px', width=0.5),
        showlegend=False,
        **pos,
    )

    fig.add_scatter(
        x = x_plot, y = y_plot,
        mode = 'lines', line = dict(color='black', width=0.5),
        showlegend = False,
        **pos,
    )

    fig.update_xaxes(
        tickvals = np.arange(-500, 500, 100),
        title_text = f'time [ms]',
        range = [bins[0]-1, bins[-1]+1],
        **pos,
    )

    fig.update_yaxes(
        range=[0, burst_offset],
        tickvals = yticks,
        ticktext = ytext,
        **pos,
    )

    sname = figure_dir  / 'raster plots' / f'{cluster_id}'

    utils.save_fig(fig, sname, display=False, verbose=False)

print('\n\t\t\t------ End Of Cell ------')

In [ ]:
'''
cluster_ids = data_io.cluster_df.index.values
blockers = ['noblocker', 'cpp', 'washout']

for cluster_id in cluster_ids:

    ec = pref_ec_dict[cluster_id]
    if ec is None:
        continue
    blocker = "noblocker"
    d_select = data_io.burst_df.query('electrode == @ec and '
                                              'Blocker == @blocker')
    d_select.sort_values('duty_cycle', inplace=True)
    duty_cycles = d_select.duty_cycle.unique()
    fmax = np.max(duty_cycles)
    print(d_select.query('duty_cycle == @fmax'))
'''

In [ ]:
##%% optimized single-panel firing rate plotting
DEBUG = False   # set to False to mute debug output

def dbg(*msg):
    if DEBUG:
        print("[DEBUG]", *msg)


x  = 0
plot_titles = [
    'PA',
    'PA + light',
]

for cluster_id in data_io.cluster_df.index.values:

    # x += 1
    # if x == 3:
    #     break

    dbg(f"\n=== Processing cluster {cluster_id} ===")

    # Create a *single panel* figure
    fig = utils.make_figure(
        width    = 1,
        height   = 1.5,
        x_domains={1: [[0.1, 0.5], [0.6, 0.9]],
                   2: [[0.1, 0.4], [0.6, 0.9]]},
        y_domains={1: [[0.6, 0.9], [0.6, 0.9]],
                   2: [[0.1, 0.4], [0.1, 0.4]]},
        subplot_titles={
            1: ['8', '12'],
            2: ['12' ,'29']
        }
    )

    # Always plot into (row=1, col=1)
    pos = dict(row=1, col=1)

    # Load data for this cluster
    dbg("Loading bootstrap data…")
    cluster_data = utils.load_obj(
        data_dir / 'bootstrapped' / f'bootstrap_{cluster_id}.pkl'
    )

    # preferred electrode
    ec = pref_ec_dict[cluster_id]
    dbg("Preferred electrode:", ec)

    if ec is None:
        dbg("Skipping cluster (no preferred electrode)")
        continue

    # Track global y-limits
    y_max = 0

    # Loop over all recordings (all curves go on same axis)
    for rec_i, rec_name in enumerate(selected_rec_names):

        dbg(f"  Recording: {rec_name}")

        # Filter data
        d_select = data_io.burst_df.query(
            'electrode == @ec and recording_name == @rec_name'
        ).copy()

        dbg("    Burst trials found:", len(d_select))

        if len(d_select) == 0:
            dbg("    WARNING: No trials match this electrode/recording")
            continue

        d_select.sort_values('duty_cycle', inplace=True)

        # Determine condition type: PA vs PA+light
        if '_light_pa_' in rec_name:
            duty_cycles = d_select.laser_duty_cycle.unique()
            dbg("    Condition detected: PA + light")
            mode = 'light'
        elif '_pa_' in rec_name:
            duty_cycles = d_select.duty_cycle.unique()
            dbg("    Condition detected: PA")
            mode = 'pa'
        else:
            dbg("    WARNING: Unrecognized condition; skipping")
            continue

        dbg("    Duty cycles:", duty_cycles)

        for dc in duty_cycles:

            if dc == 8:
                pos = dict(row=1, col=1)
            elif dc == 12:
                pos = dict(row=1, col=2)
            elif dc == 20:
                pos = dict(row=2, col=1)
            elif dc == 29:  
                pos = dict(row=2, col=2)
            dbg(f"      Duty cycle: {dc}")

            # Extract train ID + burst settings
            if mode == 'light':
                row0 = d_select.query('laser_duty_cycle == @dc').iloc[0]
                tid  = row0.train_id

                bdf  = data_io.burst_df
                frep = bdf.query('train_id == @tid').iloc[0].laser_duty_cycle
                bd   = bdf.query('train_id == @tid').iloc[0].laser_burst_duration

                
                
                clra_a = 'rgba(255, 165, 0, 0.2)'  # orange, alpha 0.2
                clr = 'rgba(255, 165, 0, 1)'      # orange, alpha 1

            elif mode == 'pa':
                row0 = d_select.query('duty_cycle == @dc').iloc[0]
                tid  = row0.train_id

                bdf  = data_io.burst_df
                frep = bdf.query('train_id == @tid').iloc[0].duty_cycle
                bd   = bdf.query('train_id == @tid').iloc[0].burst_duration

                # clr_a = clrs.duty_cycle(dc, alpha = 0.2)
                # clr   = clrs.duty_cycle(dc)
                clra_a = 'rgba(0, 128, 0, 0.2)'    # green, alpha 0.2
                clr = 'rgba(0, 128, 0, 1)'        #

            dbg("        train_id:", tid, "| burst dur:", bd)

            # Load bootstrap FR data
            bins    = cluster_data[tid]['bins']
            sp      = cluster_data[tid]['firing_rate']
            ci_low  = cluster_data[tid]['firing_rate_ci_low']
            ci_high = cluster_data[tid]['firing_rate_ci_high']

            if sp is None:
                dbg("        WARNING: No spike data for tid", tid)
                continue

            y_max = max(y_max, ci_high.max())

            # --- shaded burst window ---
            fig.add_scatter(
                x=[0, 0, bd, bd],
                y=[0, 1000, 1000, 0],
                mode='lines',
                line=dict(width=0),
                fill='toself',
                fillcolor='rgba(128, 128, 128, 0.1)',
                showlegend=False,
                **pos,
            )

            # --- confidence interval ---
            fig.add_scatter(
                x=bins,
                y=ci_low,
                line=dict(width=0),
                showlegend=False,
                **pos,
            )
            fig.add_scatter(
                x=bins,
                y=ci_high,
                mode='lines',
                line=dict(width=0),
                fill='tonexty',
                fillcolor='rgba(128, 128, 128, 0.1)',
                showlegend=False,
                **pos,
            )

            # --- firing-rate line ---
            legend_name = f"DC {dc}"
            show_leg = not any(trace.name == legend_name for trace in fig.data)

            fig.add_scatter(
                x=bins,
                y=sp,
                name=mode,
                line=dict(width=1, color=clr),
                showlegend=show_leg,
                **pos,
            )

    # Final axes formatting
    for i in range(1, 3):
        for j in range(1, 3):
            fig.update_xaxes(
                tickvals=np.arange(-500, 500, 100),
                title_text='Time (ms)',
                range=[-101, 351],
                row=i, col=j,
            )
            fig.update_yaxes(
                title_text='Firing rate (Hz)',
                tickvals=np.arange(0, np.ceil(y_max/10)*10 + 1, 10),  # nice 10-Hz spacing
                range=[0, y_max],
                row=i, col=j,
            )

    # Save figure
    sname = figure_dir / 'firing_rate_per_condition' / f'{cluster_id}'
    dbg("Saving figure to:", sname)

    utils.save_fig(fig, sname, display=False, formats=['png', 'svg'])

print("\n\nAll clusters processed.\n\t\t\t------ End Of Cell ------")


In [ ]:
##%% optimized single-panel firing rate plotting
x  = 0
plot_titles = [
    'PA',
    'PA + light',
]

for cluster_id in data_io.cluster_df.index.values:


    # Create a *single panel* figure
    fig = utils.make_figure(
        width    = 1,
        height   = 1.5,
        x_domains={1: [[0.1, 0.5], [0.6, 0.9]],
                   2: [[0.1, 0.4], [0.6, 0.9]]},
        y_domains={1: [[0.6, 0.9], [0.6, 0.9]],
                   2: [[0.1, 0.4], [0.1, 0.4]]},
        subplot_titles={
            1: ['8', '12'],
            2: ['12' ,'29']
        }
    )

    # Always plot into (row=1, col=1)
    pos = dict(row=1, col=1)

    # Load data for this cluster
    cluster_data = utils.load_obj(
        data_dir / 'bootstrapped' / f'bootstrap_{cluster_id}.pkl'
    )

    # preferred electrode
    ec = pref_ec_dict[cluster_id]

    if ec is None:
        continue

    # Track global y-limits
    y_max = 0

    # Loop over all recordings (all curves go on same axis)
    for rec_i, rec_name in enumerate(selected_rec_names):

        # Filter data
        d_select = data_io.burst_df.query(
            'electrode == @ec and recording_name == @rec_name'
        ).copy()

        if len(d_select) == 0:
            continue

        d_select.sort_values('duty_cycle', inplace=True)

        # Determine condition type: PA vs PA+light
        if '_light_pa_' in rec_name:
            duty_cycles = d_select.laser_duty_cycle.unique()
            mode = 'light'
        elif '_pa_' in rec_name:
            duty_cycles = d_select.duty_cycle.unique()
            mode = 'pa'
        else:
            continue

        for dc in duty_cycles:

            if dc == 8:
                pos = dict(row=1, col=1)
            elif dc == 12:
                pos = dict(row=1, col=2)
            elif dc == 20:
                pos = dict(row=2, col=1)
            elif dc == 29:  
                pos = dict(row=2, col=2)

            # Extract train ID + burst settings
            if mode == 'light':
                row0 = d_select.query('laser_duty_cycle == @dc').iloc[0]
                tid  = row0.train_id

                bdf  = data_io.burst_df
                frep = bdf.query('train_id == @tid').iloc[0].laser_duty_cycle
                bd   = bdf.query('train_id == @tid').iloc[0].laser_burst_duration

                
                
                clra_a = 'rgba(255, 165, 0, 0.2)'  # orange, alpha 0.2
                clr = 'rgba(255, 165, 0, 1)'      # orange, alpha 1

            elif mode == 'pa':
                row0 = d_select.query('duty_cycle == @dc').iloc[0]
                tid  = row0.train_id

                bdf  = data_io.burst_df
                frep = bdf.query('train_id == @tid').iloc[0].duty_cycle
                bd   = bdf.query('train_id == @tid').iloc[0].burst_duration

                # clr_a = clrs.duty_cycle(dc, alpha = 0.2)
                # clr   = clrs.duty_cycle(dc)
                clra_a = 'rgba(0, 128, 0, 0.2)'    # green, alpha 0.2
                clr = 'rgba(0, 128, 0, 1)'        #


            # Load bootstrap FR data
            bins    = cluster_data[tid]['bins']
            sp      = cluster_data[tid]['firing_rate']
            ci_low  = cluster_data[tid]['firing_rate_ci_low']
            ci_high = cluster_data[tid]['firing_rate_ci_high']

            if sp is None:
                continue

            y_max = max(y_max, ci_high.max())

            # --- shaded burst window ---
            fig.add_scatter(
                x=[0, 0, bd, bd],
                y=[0, 1000, 1000, 0],
                mode='lines',
                line=dict(width=0),
                fill='toself',
                fillcolor='rgba(128, 128, 128, 0.1)',
                showlegend=False,
                **pos,
            )

            # --- confidence interval ---
            fig.add_scatter(
                x=bins,
                y=ci_low,
                line=dict(width=0),
                showlegend=False,
                **pos,
            )
            fig.add_scatter(
                x=bins,
                y=ci_high,
                mode='lines',
                line=dict(width=0),
                fill='tonexty',
                fillcolor='rgba(128, 128, 128, 0.1)',
                showlegend=False,
                **pos,
            )

            # --- firing-rate line ---
            legend_name = f"DC {dc}"
            show_leg = not any(trace.name == legend_name for trace in fig.data)

            fig.add_scatter(
                x=bins,
                y=sp,
                name=mode,
                line=dict(width=1, color=clr),
                showlegend=show_leg,
                **pos,
            )

    # Final axes formatting
    for i in range(1, 3):
        for j in range(1, 3):
            fig.update_xaxes(
                tickvals=np.arange(-500, 500, 100),
                title_text='Time (ms)',
                range=[-101, 351],
                row=i, col=j,
            )
            fig.update_yaxes(
                title_text='Firing rate (Hz)',
                tickvals=np.arange(0, np.ceil(y_max/10)*10 + 1, 10),  # nice 10-Hz spacing
                range=[0, y_max],
                row=i, col=j,
            )

    # Save figure
    sname = figure_dir / 'test' / f'{cluster_id}'

    utils.save_fig(fig, sname, display=False, formats=['png', 'svg'])

print("\n\nAll clusters processed.\n\t\t\t------ End Of Cell ------")


# Test WIP Audrey
### Generating a summary dataframe

In [26]:
''' Test WIP '''
##%% optimized single-panel firing rate plotting
x  = 0
pa_lat = []
light_lat = []

for cluster_id in data_io.cluster_df.index.values:

    # Load data for this cluster
    cluster_data = utils.load_obj(
        data_dir / 'bootstrapped' / f'bootstrap_{cluster_id}.pkl'
    )

    # preferred electrode
    ec = pref_ec_dict[cluster_id]

    if ec is None:
        continue

    # Track global y-limits
    y_max = 0

    # Loop over all recordings (all curves go on same axis)
    for rec_i, rec_name in enumerate(selected_rec_names):

        # Filter data
        d_select = data_io.burst_df.query(
            'electrode == @ec and recording_name == @rec_name'
        ).copy()

        if len(d_select) == 0:
            continue

        d_select.sort_values('duty_cycle', inplace=True)

        # Determine condition type: PA vs PA+light
        #if '_PADMD_' in rec_name:
        if '_light_pa_' in rec_name:
            duty_cycles = d_select.laser_duty_cycle.unique()
            mode = 'light'
        #elif '_PA_' in rec_name:
        elif '_pa_' in rec_name:
            duty_cycles = d_select.duty_cycle.unique()
            mode = 'pa'
        else:
            continue

        for dc in duty_cycles:
            # Extract train ID + burst settings
            row0    = d_select.query('laser_duty_cycle == @dc').iloc[0]
            tid     = row0.train_id
            lat     = cells_df.xs(cluster_id)[tid, 'response_latency']
            repfr   = cells_df.xs(cluster_id)[tid, 'response_firing_rate']
            reptype = cells_df.xs(cluster_id)[tid, 'response_type']

            if not np.isnan(lat) :
                if mode == 'light':
                    light_lat.append([cluster_id, ec, dc, lat, repfr, reptype])

                elif mode == 'pa':
                    pa_lat.append([cluster_id, ec, dc, lat, repfr, reptype])

# Summary df for pa responses
df_pa_lat = pd.DataFrame(pa_lat, columns=['cluster_id', 'pref_ec', 'dc', 'pa_response_latency', 'pa_rep_fr','pa_rep_type'])
df_pa_lat.set_index(['cluster_id', 'pref_ec','dc'], inplace = True)

# Summary df for light + pa responses
df_light_lat = pd.DataFrame(light_lat, columns=['cluster_id', 'pref_ec', 'dc', 'light_response_latency', 'light_rep_fr','light_rep_type'])
df_light_lat.set_index(['cluster_id', 'pref_ec','dc'], inplace = True)

# Summary df
lat_df = pd.concat([df_pa_lat, df_light_lat], axis = 1, join = 'outer')

print("\n\nAll clusters processed.\nSummary df generated.\n\t\t\t------ End Of Cell ------")




All clusters processed.
Summary df generated.
			------ End Of Cell ------


### Plot PA vs light responses

In [24]:
df = lat_df.reset_index()

dc_values = sorted(df["dc"].unique())

# Remove dc = 37
df = df[df["dc"] != 37]

dc_values = sorted(df["dc"].unique())
clusters = sorted(df["cluster_id"].unique())

# -------------------
# Figure layout
# -------------------
n_cols = 2
n_rows = int(np.ceil(len(dc_values) / n_cols))

subplot_titles = {
    r + 1: [f"DC = {dc_values[r*n_cols + c]}" if r*n_cols + c < len(dc_values) else ""
            for c in range(n_cols)]
    for r in range(n_rows)
}

fig = utils.simple_fig(
    n_cols=n_cols,
    n_rows=n_rows,
    width=1.4,
    height=0.9 * n_rows,
    subplot_titles=subplot_titles,
    # equal_width_height="x",   # enforce square panels
)

# -------------------
# Axis limits (shared origin at 0)
# -------------------
max_val = max(
    df["pa_response_latency"].max(),
    df["light_response_latency"].max()
)

axis_min = 0
axis_max = max_val * 1.05

# First axes explicitly
fig.update_layout(
    xaxis=dict(range=[axis_min, axis_max]),
    yaxis=dict(range=[axis_min, axis_max], scaleanchor="x", scaleratio=1),
)

# Match all others
fig.update_xaxes(matches="x")
fig.update_yaxes(matches="y", scaleanchor="x", scaleratio=1)

# -------------------
# Plotting
# -------------------
for i, dc in enumerate(dc_values):
    row = i // n_cols + 1
    col = i % n_cols + 1

    sub = df[df["dc"] == dc]

    # per-cluster scatter
    for ci, cluster_id in enumerate(clusters):
        cd = sub[sub["cluster_id"] == cluster_id]
        if cd.empty:
            continue

        color = utils.interp_color(len(clusters), ci, ("qualitative", "Plotly"), alpha=0.9)

        fig.add_trace(
            go.Scatter(
                x=cd["pa_response_latency"],
                y=cd["light_response_latency"],
                mode="markers",
                marker=dict(color=color, size=5),
                name=f"cluster {cluster_id}",
                legendgroup=str(cluster_id),
                showlegend=(i == 0),
            ),
            row=row,
            col=col,
        )

    # identity line
    fig.add_trace(
        go.Scatter(
            x=[axis_min, axis_max],
            y=[axis_min, axis_max],
            mode="lines",
            line=dict(color="black", dash="dash", width=1),
            showlegend=False,
        ),
        row=row,
        col=col,
    )

# -------------------
# Axis labels
# -------------------
fig.update_xaxes(tickmode="auto", showticklabels=True, title_text="PA response latency")
fig.update_yaxes(tickmode="auto", showticklabels=True, title_text="Light response latency")

fig.update_layout(
    xaxis=dict(range=[0, axis_max], zeroline=True),
    yaxis=dict(range=[0, axis_max], scaleanchor="x"),
)

# Match all others
fig.update_xaxes(matches="x")
fig.update_yaxes(matches="y", scaleanchor="x", scaleratio=1)

fig.show()
# sname = figure_dir / f'{session_id}_test_5v9'
# save_fig(fig, sname, formats=['png', 'svg'], scale=3)

ZeroDivisionError: float division by zero

In [32]:
df = lat_df.reset_index()

dc_values = sorted(df["dc"].unique())

if len(dc_values) == 0:
    raise ValueError(
        "No data left after filtering by rep_case. "
        "Check pa_rep_type / light_rep_type values in the CSV."
    )

# Remove dc = 37
df = df[df["dc"] != 37]

# -----------------------
# Define case categories
# -----------------------
def normalize(v):
    if pd.isna(v):
        return "nan"
    return str(v).lower()

def rep_case(row):
    pa = normalize(row["pa_rep_type"])
    li = normalize(row["light_rep_type"])
    return f"{pa}|{li}"

df["rep_case"] = df.apply(rep_case, axis=1)

case_labels = {
    "inhibited|inhibited": "PA inh / Light inh",
    "inhibited|excited":   "PA inh / Light exc",
    "inhibited|nan":       "PA inh / Light nan",
    "excited|inhibited":   "PA exc / Light inh",
    "excited|excited":     "PA exc / Light exc",
    "excited|nan":         "PA exc / Light nan",
    "nan|inhibited":       "PA nan / Light inh",
    "nan|excited":         "PA nan / Light exc",
}

df = df[df["rep_case"].isin(case_labels.keys())]

cases = list(case_labels.keys())

# -----------------------
# Subplot layout
# -----------------------
dc_values = sorted(df["dc"].unique())
n_cols = 2
n_rows = int(np.ceil(len(dc_values) / n_cols))

subplot_titles = {
    r + 1: [
        f"DC = {dc_values[r*n_cols + c]}" if r*n_cols + c < len(dc_values) else ""
        for c in range(n_cols)
    ]
    for r in range(n_rows)
}

fig = utils.simple_fig(
    n_cols=n_cols,
    n_rows=n_rows,
    width=1.4,
    height=0.9 * n_rows,
    subplot_titles=subplot_titles,
)

# -----------------------
# Axis limits (shared)
# -----------------------
max_val = max(
    df["pa_response_latency"].max(),
    df["light_response_latency"].max()
)

axis_min = 0
axis_max = max_val * 1.05

fig.update_layout(
    xaxis=dict(range=[axis_min, axis_max], zeroline=True),
    yaxis=dict(range=[axis_min, axis_max], scaleanchor="x", scaleratio=1, zeroline=True),
)

fig.update_xaxes(matches="x", tickmode="auto", showticklabels=True)
fig.update_yaxes(matches="y", scaleanchor="x", scaleratio=1, tickmode="auto", showticklabels=True)

# -----------------------
# Colors for cases
# -----------------------
case_colors = {}
for i, case in enumerate(cases):
    case_colors[case] = utils.interp_color(
        nsteps=len(cases),
        step_nr=i,
        scalename=("qualitative", "Plotly"),
        alpha=0.9
    )

# -----------------------
# Plot
# -----------------------
for i, dc in enumerate(dc_values):
    row = i // n_cols + 1
    col = i % n_cols + 1

    sub = df[df["dc"] == dc]

    for case in cases:
        cd = sub[sub["rep_case"] == case]
        if cd.empty:
            continue

        fig.add_trace(
            go.Scatter(
                x=cd["pa_response_latency"],
                y=cd["light_response_latency"],
                mode="markers",
                marker=dict(color=case_colors[case], size=5),
                name=case_labels[case],
                legendgroup=case,
                showlegend=(i == 0),
                hovertemplate=(
                    "PA: %{x}<br>"
                    "Light: %{y}<br>"
                    f"{case_labels[case]}<extra></extra>"
                ),
            ),
            row=row,
            col=col,
        )

    # Identity line
    fig.add_trace(
        go.Scatter(
            x=[axis_min, axis_max],
            y=[axis_min, axis_max],
            mode="lines",
            line=dict(color="black", dash="dash", width=1),
            showlegend=False,
        ),
        row=row,
        col=col,
    )

# Axis labels
fig.update_xaxes(title_text="PA response latency")
fig.update_yaxes(title_text="Light response latency")

fig.show()
sname = figure_dir / f'{session_id}_test'
save_fig(fig, sname, formats=['png'], scale=3)


saved: /media/aleong/Audrey-experiments/dataset/251015/251015_test.png
displaying figure


sh: 1: start: not found
